# D2 Fresh-Holdout Confirmation Check

Every experiment so far (A through D2, the model comparison, the 5-fold CV, and the interpretability analysis) used the same `random_state=42` train/validation/test split. This notebook is a supplementary confirmation check, not a new model-selection round: it re-evaluates the three already-selected D2 model configurations on a differently-seeded partition of the same population, to see whether the results hold up outside the one specific split used everywhere else.

Nothing is retuned here. The three model configurations and their locked classification thresholds are copied directly from `Model_Comparison_D2.ipynb`. Only the data partition changes, using `random_state=137` instead of `42`.

## Population and target

Same as Model_Comparison_D2: `CCC_05` in {1, 2}, target = `CCC_05 == 1`.

In [1]:
import pandas as pd
import numpy as np

pumf = pd.read_csv("../Data_Données/pumf_cchs.csv")

print("PUMF shape:", pumf.shape)

PUMF shape: (67079, 255)


In [2]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts().sort_index())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


## D2 feature set

Same 9 features as Model_Comparison_D2.

In [3]:
FEATURES = [
    "DHHGAGE",
    "DHH_SEX",
    "EDDVH3",
    "BMI_CLASS",
    "INCDGHH",
    "SDCDGIMM",
    "GEOGPRV",
    "CCC_80",
    "CCC_90"
]

print("Number of features:", len(FEATURES))
print(FEATURES)

Number of features: 9
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80', 'CCC_90']


In [4]:
# Same BMI harmonization as C, D1, D2, Model_Comparison_D2
model_data["BMI_CLASS"] = np.nan

youth_mask = model_data["DHHGAGE"] == 1
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])

model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [5]:
SPECIAL_CODES = {
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "BMI_CLASS": [6, 9],
    "INCDGHH": [9],
    "SDCDGIMM": [9],
    "GEOGPRV": [],
    "CCC_80": [9],
    "CCC_90": [9]
}

def apply_special_codes(df, special_codes):
    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

print("Missing values after special-code handling:")
print(clean_model_data[FEATURES].isna().sum())

Missing values after special-code handling:
DHHGAGE         0
DHH_SEX         0
EDDVH3       2276
BMI_CLASS    3144
INCDGHH       947
SDCDGIMM      835
GEOGPRV         0
CCC_80        494
CCC_90        155
dtype: int64


## Fresh split: random_state=137

Everything else in the project uses `random_state=42`. This notebook uses `random_state=137`, an unused seed, for the train_val/test split and the train/validation split, so the fresh test portion contains different respondents than every other notebook. Split proportions (80/20, then 80/20 again) are unchanged, so sizes should come out close to the original 42,394 / 10,599 / 13,249.

In [6]:
from sklearn.model_selection import train_test_split

FRESH_RANDOM_STATE = 137

fresh_train_val_idx, fresh_test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    stratify=model_data["target"],
    random_state=FRESH_RANDOM_STATE
)

fresh_train_idx, fresh_val_idx = train_test_split(
    fresh_train_val_idx,
    test_size=0.20,
    stratify=model_data.loc[fresh_train_val_idx, "target"],
    random_state=FRESH_RANDOM_STATE
)

print("Fresh split random_state:", FRESH_RANDOM_STATE)
print("Fresh training:", len(fresh_train_idx))
print("Fresh validation:", len(fresh_val_idx))
print("Fresh test:", len(fresh_test_idx))

Fresh split random_state: 137
Fresh training: 42394
Fresh validation: 10599
Fresh test: 13249


In [7]:
X_train = clean_model_data.loc[fresh_train_idx, FEATURES]
X_val = clean_model_data.loc[fresh_val_idx, FEATURES]
X_test = clean_model_data.loc[fresh_test_idx, FEATURES]

y_train = model_data.loc[fresh_train_idx, "target"]
y_val = model_data.loc[fresh_val_idx, "target"]
y_test = model_data.loc[fresh_test_idx, "target"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (42394, 9) (42394,)
Validation: (10599, 9) (10599,)
Test: (13249, 9) (13249,)


The validation portion above is only produced for consistency with the original two-stage split procedure. It is **not** used to reselect a threshold or hyperparameters in this notebook — the locked settings from Model_Comparison_D2 are applied as-is.

## Preprocessing

Same categorical pipeline as Model_Comparison_D2, fitted on the fresh training portion only.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, FEATURES)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (42394, 34)
Processed validation shape: (10599, 34)
Processed test shape: (13249, 34)


## Fixed models and locked thresholds

These are exactly the configurations and thresholds already selected in `Model_Comparison_D2.ipynb`. No search, no retuning, no threshold recalculation.

| Model | Configuration | Locked threshold |
|---|---|---|
| Logistic Regression | `class_weight="balanced"`, `max_iter=1000`, `random_state=42` | 0.68 |
| Random Forest | `n_estimators=200`, `max_depth=10`, `class_weight="balanced"`, `random_state=42` | 0.68 |
| Gradient Boosting | `n_estimators=200`, `learning_rate=0.05`, `max_depth=3`, `random_state=42`, `sample_weight` balanced | 0.70 |

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

LOG_REG_THRESHOLD = 0.68
RF_THRESHOLD = 0.68
GB_THRESHOLD = 0.70

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(X_train_processed, y_train)

random_forest = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1
)
random_forest.fit(X_train_processed, y_train)

train_sample_weight = compute_sample_weight("balanced", y_train)
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
)
gradient_boosting.fit(X_train_processed, y_train, sample_weight=train_sample_weight)

print("All three models fitted on the fresh training portion.")

All three models fitted on the fresh training portion.


## Fresh test evaluation

The fresh test portion is evaluated once, using the locked thresholds above.

In [10]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, log_loss
)

def evaluate_on_fresh_test(model, threshold, model_name):
    train_prob = model.predict_proba(X_train_processed)[:, 1]
    test_prob = model.predict_proba(X_test_processed)[:, 1]

    test_pred = (test_prob >= threshold).astype(int)

    cm = confusion_matrix(y_test, test_pred)
    tn, fp, fn, tp = cm.ravel()

    result = {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, test_prob),
        "pr_auc": average_precision_score(y_test, test_prob),
        "train_log_loss": log_loss(y_train, train_prob),
        "test_log_loss": log_loss(y_test, test_prob),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }
    return result, cm

log_reg_result, log_reg_cm = evaluate_on_fresh_test(log_reg, LOG_REG_THRESHOLD, "Logistic Regression")
rf_result, rf_cm = evaluate_on_fresh_test(random_forest, RF_THRESHOLD, "Random Forest")
gb_result, gb_cm = evaluate_on_fresh_test(gradient_boosting, GB_THRESHOLD, "Gradient Boosting")

fresh_results_df = pd.DataFrame([log_reg_result, rf_result, gb_result])
print(fresh_results_df.round(4))

                 model  threshold  accuracy  precision  recall      f1  \
0  Logistic Regression       0.68    0.8298     0.2861  0.5888  0.3851   
1        Random Forest       0.68    0.8404     0.2978  0.5621  0.3894   
2    Gradient Boosting       0.70    0.8412     0.2968  0.5513  0.3859   

   roc_auc  pr_auc  train_log_loss  test_log_loss  true_negatives  \
0   0.8292  0.3008          0.5350         0.5276           10288   
1   0.8298  0.2978          0.4974         0.5044           10461   
2   0.8323  0.3001          0.5246         0.5188           10484   

   false_positives  false_negatives  true_positives  
0             1762              493             706  
1             1589              525             674  
2             1566              538             661  


## Confusion matrices (fresh test portion)

In [11]:
print("Logistic Regression")
print(log_reg_cm)

print("\nRandom Forest")
print(rf_cm)

print("\nGradient Boosting")
print(gb_cm)

Logistic Regression
[[10288  1762]
 [  493   706]]

Random Forest
[[10461  1589]
 [  525   674]]

Gradient Boosting
[[10484  1566]
 [  538   661]]


## Save results

In [12]:
fresh_results_to_save = fresh_results_df.copy()
fresh_results_to_save["split_random_state"] = FRESH_RANDOM_STATE

fresh_results_to_save.to_csv("d2_fresh_holdout_results.csv", index=False)

print("Saved: d2_fresh_holdout_results.csv")
fresh_results_to_save

Saved: d2_fresh_holdout_results.csv


,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,train_log_loss,test_log_loss,true_negatives,false_positives,false_negatives,true_positives,split_random_state
0,Logistic Regression,0.68,0.829798,0.286062,0.588824,0.385056,0.829204,0.300773,0.535031,0.527644,10288,1762,493,706,137
1,Random Forest,0.68,0.840441,0.297835,0.562135,0.389370,0.829754,0.297807,0.497355,0.504355,10461,1589,525,674,137
2,Gradient Boosting,0.70,0.841196,0.296812,0.551293,0.385873,0.832291,0.300087,0.524622,0.518813,10484,1566,538,661,137


## Comparison with the original Model_Comparison_D2 test results

The table below places the fresh-holdout numbers (random_state=137) next to the original test results (random_state=42) from Model_Comparison_D2, purely for comparison. This comparison does not change any model, threshold, or feature decision — the D2 configuration remains whatever was already locked.

| | LR (orig) | LR (fresh) | RF (orig) | RF (fresh) | GB (orig) | GB (fresh) |
|---|---|---|---|---|---|---|
| Accuracy | 0.825 | 0.830 | 0.840 | 0.840 | 0.840 | 0.841 |
| Precision | 0.269 | 0.286 | 0.281 | 0.298 | 0.285 | 0.297 |
| Recall | 0.542 | 0.589 | 0.492 | 0.562 | 0.507 | 0.551 |
| F1 | 0.360 | 0.385 | 0.357 | 0.389 | 0.365 | 0.386 |
| ROC-AUC | 0.814 | 0.829 | 0.811 | 0.830 | 0.815 | 0.832 |
| PR-AUC | 0.289 | 0.301 | 0.276 | 0.298 | 0.286 | 0.300 |
| Test log loss | 0.528 | 0.528 | 0.506 | 0.504 | 0.520 | 0.519 |

All three models score somewhat better on this fresh test partition than on the original one, most noticeably on recall (up 4-7 points) and F1 (up about 2-3 points), with ROC-AUC and PR-AUC also higher across the board. This means the original test partition was, if anything, a slightly harder split for these models, not an artificially favorable one.

More importantly for this check, the pattern between the three models is preserved. Test log loss keeps the exact same ranking in both splits (Random Forest lowest, then Gradient Boosting, then Logistic Regression), with nearly identical values (e.g. Random Forest 0.506 vs 0.504). PR-AUC also keeps the same ranking (Logistic Regression highest, then Gradient Boosting, then Random Forest) in both splits. F1 and ROC-AUC show minor reordering among the three models (Random Forest and Gradient Boosting swap for the top spot), but the gaps between models are small in both splits (a few thousandths to about a hundredth), consistent with the "no model clearly dominates" conclusion already reached in Model_Comparison_D2.

Overall, this fresh partition does not overturn anything: the three D2 model configurations behave similarly to how they behaved on the original test set, and none of them falls apart or reverses its relationship with the others on a differently-seeded holdout.

## Note on scope

This is a confirmation check on one alternative partition, not a new model-selection round and not a full stability study. The original locked D2 models, thresholds, and reported test results in Model_Comparison_D2 are unchanged by this notebook.